# Week 1 exercise (generic)

Reusable helpers for two tasks from the original exercise:

1. **Explain a technical question or snippet of code** (OpenAI or Ollama, streaming).
2. **Summarize a YouTube video** from its transcript (OpenAI or Ollama, streaming).

Change the inputs in the **Usage** cells. The original notebook is unchanged.

In [ ]:
import os

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from youtube_transcript_api import YouTubeTranscriptApi

In [ ]:
# Defaults — override per call if you want a different model or endpoint

MODEL_GPT = "gpt-4o-mini"
MODEL_OLLAMA = "llama3.2"
OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_HEALTH_URL = "http://localhost:11434"

DEFAULT_CODE_SYSTEM_PROMPT = """
You are a helpful assistant that can explain code to technical and non-technical audiences.
Explain the code in a way that is easy to understand.
You are also an expert in computer science, programming, LLMs, and their capabilities.
""".strip()

DEFAULT_SUMMARY_SYSTEM_PROMPT = """
You are a helpful assistant that can summarize YouTube transcripts.
Summarise the transcript in a way that is easy to understand.
Write the summary in bullet points.
If useful, break the summary into sections (for example: overview, key points).
""".strip()

In [ ]:
def load_openai_api_key() -> str | None:
    """Load OPENAI_API_KEY from the environment (and .env if present)."""
    load_dotenv(override=True)
    api_key = os.getenv("OPENAI_API_KEY")
    if api_key and api_key.startswith("sk-") and len(api_key) > 10:
        print("API key looks good so far")
        return api_key
    print(
        "There might be a problem with your API key? "
        "Set OPENAI_API_KEY in your environment or a .env file."
    )
    return api_key


def get_openai_client(api_key: str | None = None) -> OpenAI:
    """OpenAI client for cloud models (gpt-4o-mini, etc.)."""
    return OpenAI(api_key=api_key or os.getenv("OPENAI_API_KEY"))


def get_ollama_client(
    base_url: str = OLLAMA_BASE_URL,
    health_url: str = OLLAMA_HEALTH_URL,
) -> OpenAI:
    """OpenAI-compatible client pointing at a local Ollama server."""
    try:
        requests.get(health_url, timeout=5).raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            f"Ollama does not appear to be running at {health_url}. "
            "Start Ollama and pull the model first (e.g. `ollama pull llama3.2`)."
        ) from exc
    return OpenAI(base_url=base_url, api_key="ollama")

In [ ]:
def build_messages(
    user_content: str,
    system_prompt: str,
    user_prompt_template: str | None = None,
) -> list[dict[str, str]]:
    """Build a standard chat-completions message list."""
    if user_prompt_template:
        user_content = user_prompt_template.format(content=user_content)
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content},
    ]


def stream_chat(
    client: OpenAI,
    messages: list[dict[str, str]],
    model: str,
    render: bool = True,
) -> str:
    """Stream a chat completion and optionally render Markdown in the notebook."""
    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True,
    )
    response = ""
    display_handle = None
    if render:
        display_handle = display(Markdown(response), display_id=True)

    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        if render and display_handle is not None:
            update_display(Markdown(response), display_id=display_handle.display_id)
    return response


def ask_llm(
    messages: list[dict[str, str]],
    provider: str = "openai",
    model: str | None = None,
    client: OpenAI | None = None,
    render: bool = True,
) -> str:
    """Send messages to OpenAI or Ollama and stream the reply.

    provider: "openai" or "ollama"
    """
    provider = provider.lower().strip()
    if client is None:
        if provider == "openai":
            client = get_openai_client()
            model = model or MODEL_GPT
        elif provider == "ollama":
            client = get_ollama_client()
            model = model or MODEL_OLLAMA
        else:
            raise ValueError(f"Unknown provider: {provider!r}. Use 'openai' or 'ollama'.")
    elif model is None:
        model = MODEL_GPT if provider == "openai" else MODEL_OLLAMA

    return stream_chat(client, messages, model=model, render=render)

## Code / question explainer

`explain()` takes any question or code snippet and streams an explanation from the chosen provider.

In [ ]:
def explain(
    question: str,
    provider: str = "openai",
    model: str | None = None,
    system_prompt: str = DEFAULT_CODE_SYSTEM_PROMPT,
    render: bool = True,
) -> str:
    """Explain a technical question or code snippet."""
    messages = build_messages(
        question.strip(),
        system_prompt=system_prompt,
        user_prompt_template="Please explain what this code does and why:\n{content}",
    )
    return ask_llm(messages, provider=provider, model=model, render=render)

In [ ]:
# Usage: paste any question or code here, then pick a provider

QUESTION = """
Please explain what this code does and why:
def fibonacci(n: int) -> list[int]:
    #Return the first n Fibonacci numbers.
    if n <= 0:
        return []
    if n == 1:
        return [0]
    seq = [0, 1]
    while len(seq) < n:
        seq.append(seq[-1] + seq[-2])
    return seq
""".strip()

load_openai_api_key()

print("--- OpenAI ---")
explain(QUESTION, provider="openai")

In [ ]:
print("--- Ollama ---")
explain(QUESTION, provider="ollama")

## YouTube transcript summarizer

`summarize_youtube()` fetches the transcript for a video ID (or URL) and streams a structured summary.

In [ ]:
def extract_youtube_video_id(video: str) -> str:
    """Accept a raw video id or a youtube.com / youtu.be URL."""
    video = video.strip()
    if "youtube.com" in video and "v=" in video:
        return video.split("v=")[1].split("&")[0]
    if "youtu.be/" in video:
        return video.split("youtu.be/")[1].split("?")[0]
    return video


def fetch_youtube_transcript(
    video: str,
    languages: list[str] | None = None,
) -> str:
    """Download a YouTube transcript and return it as plain text."""
    video_id = extract_youtube_video_id(video)
    languages = languages or ["en"]
    api = YouTubeTranscriptApi()
    transcript = api.list(video_id).find_generated_transcript(languages).fetch()
    return " ".join(snippet.text for snippet in transcript)


def summarize_youtube(
    video: str,
    provider: str = "openai",
    model: str | None = None,
    languages: list[str] | None = None,
    system_prompt: str = DEFAULT_SUMMARY_SYSTEM_PROMPT,
    render: bool = True,
) -> str:
    """Fetch a YouTube transcript and summarize it with the chosen LLM."""
    text = fetch_youtube_transcript(video, languages=languages)
    messages = build_messages(
        text,
        system_prompt=system_prompt,
        user_prompt_template="Please summarize the following transcript:\n{content}",
    )
    return ask_llm(messages, provider=provider, model=model, render=render)

In [ ]:
# Usage: set a video id or full URL

VIDEO = "NvgMnsaKmZU"  # or "https://www.youtube.com/watch?v=NvgMnsaKmZU"

print("--- OpenAI ---")
summarize_youtube(VIDEO, provider="openai")

In [ ]:
print("--- Ollama ---")
summarize_youtube(VIDEO, provider="ollama")